# Post-Stage B Experiments — Google Colab GPU Runner

Thin orchestration notebook calling reusable modules in `src/bangla_smish/`.

### Execution Policy
- **Stage B synthetic generation is permanently stopped.**
- **Default mode is `RUN_MODE = "gpu_smoke_test"`** so running the notebook first validates CUDA, dependencies, Drive mounting, manifest verification, checkpointing, and safe resume on a tiny subset before consuming significant compute units.
- Only switch `RUN_MODE = "experiments_abcd"` after the GPU verification test succeeds and you explicitly approve starting **Experiments A, B, C, and D**:
  - **Experiment A**: Unbalanced Real Baseline (`eligible_train`, `N = 3,999`)
  - **Experiment B**: Original-Only Balanced (`eligible_train` downsampled to minority class per fold, `942/class` full-pool)
  - **Experiment C**: Clean Stage B Promotional Augmentation (`eligible_train + Batches 1–7`, `N = 4,344`, synthetic train-only)
  - **Experiment D**: Full Stage B Balanced Pool (`eligible_train + Pilot 300 + Batches 1–7`, `N = 4,644`, synthetic train-only)
- **Security**: Never place tokens, service account keys, or credentials in this notebook or Git. Google Drive is mounted interactively via `google.colab.drive.mount` only after your authorization.

In [ ]:
#@title 1. Runtime Configuration & Safety Mode
RUN_MODE = 'gpu_smoke_test'  #@param ['gpu_smoke_test', 'experiments_abcd']
REQUIRE_GPU = True  #@param {type:'boolean'}
USE_GOOGLE_DRIVE = True  #@param {type:'boolean'}
DRIVE_WORKSPACE_REL = 'bangla_smish_post_stage_b'  #@param {type:'string'}
RESUME = True  #@param {type:'boolean'}
RUN_INTERRUPTION_RESUME_CHECK = True  #@param {type:'boolean'}

print(f'Configured RUN_MODE={RUN_MODE!r}, USE_GOOGLE_DRIVE={USE_GOOGLE_DRIVE}, RESUME={RESUME}')
if RUN_MODE == 'experiments_abcd':
    print('NOTE: Full Experiments A, B, C, D mode selected. Ensure GPU smoke test has already passed and been approved.')

In [ ]:
#@title 2. Verify Runtime Environment & GPU Availability
import os, platform, shutil, subprocess, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
print('Python:', sys.version.split()[0], '| Platform:', platform.platform(), '| IN_COLAB:', IN_COLAB)

if shutil.which('nvidia-smi'):
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip())
else:
    print('nvidia-smi not found in PATH.')

import torch
cuda_ok = torch.cuda.is_available()
print('PyTorch:', torch.__version__, '| CUDA available:', cuda_ok)
if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| VRAM: {props.total_memory / (1024**3):.2f} GB')
elif REQUIRE_GPU:
    raise RuntimeError('CUDA GPU is required. In Colab: Runtime -> Change runtime type -> T4 / L4 / A100 GPU.')

disk = shutil.disk_usage('/content' if Path('/content').exists() else '.')
print(f'Free local disk: {disk.free / (1024**3):.2f} GB')

In [ ]:
#@title 3. Authorize Google Drive Mount & Locate Project Workspace
from pathlib import Path
import sys

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    # Explicit interactive OAuth authorization by the user; no credentials stored in code.
    drive.mount('/content/drive', force_remount=False)
    WORKSPACE_ROOT = Path('/content/drive/MyDrive') / DRIVE_WORKSPACE_REL
else:
    WORKSPACE_ROOT = Path('/content/bangla_smish_post_stage_b')

SRC_DIR = WORKSPACE_ROOT / 'src'
BUNDLE_DIR = WORKSPACE_ROOT / 'data' / 'processed' / 'post_stage_b_bundle'
CONFIG_DIR = WORKSPACE_ROOT / 'experiments' / 'colab' / 'configs'
ARTIFACTS_DIR = WORKSPACE_ROOT / 'artifacts' / RUN_MODE
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print('WORKSPACE_ROOT:', WORKSPACE_ROOT)
print('BUNDLE_DIR exists:', BUNDLE_DIR.exists(), '| ARTIFACTS_DIR:', ARTIFACTS_DIR)
if not SRC_DIR.exists() or not BUNDLE_DIR.exists():
    raise FileNotFoundError(
        f'Missing src or dataset bundle under {WORKSPACE_ROOT}. '
        'Run scripts/prepare_post_stage_b_bundle.py --export-drive-kit locally and upload it to Google Drive.'
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
#@title 4. Install Compatible Dependencies (Preserving Colab CUDA PyTorch)
import importlib, subprocess, sys

REQUIRED_PACKAGES = [
    ('transformers', 'transformers>=4.44,<5.0'),
    ('datasets', 'datasets>=2.20'),
    ('accelerate', 'accelerate>=0.33'),
    ('safetensors', 'safetensors>=0.4'),
    ('sklearn', 'scikit-learn>=1.4'),
    ('pyarrow', 'pyarrow>=15.0'),
]
to_install = []
for mod_name, spec in REQUIRED_PACKAGES:
    try:
        mod = importlib.import_module(mod_name)
        print(f'{mod_name}: {getattr(mod, "__version__", "ok")}')
    except ImportError:
        to_install.append(spec)

if to_install:
    print('Installing missing packages:', to_install)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install])
else:
    print('All required packages are already available.')

In [ ]:
#@title 5. Load & Verify Dataset Manifest + Experiment Config
import json
from bangla_smish.post_stage_b_data import verify_manifest
from bangla_smish.post_stage_b_runner import collect_environment_info, load_config, seed_everything

manifest = verify_manifest(BUNDLE_DIR, verify_checksums=True)
print('Verified manifest_version:', manifest['manifest_version'])
print('Integrity checks:', json.dumps(manifest['integrity_checks'], indent=2))
print('Cohort counts:', json.dumps({k: v.get('by_class', v) for k, v in manifest['cohort_counts'].items()}, indent=2))

cfg_filename = 'gpu_smoke_test_config.json' if RUN_MODE == 'gpu_smoke_test' else 'post_stage_b_abcd_config.json'
cfg = load_config(CONFIG_DIR / cfg_filename)
seed_everything(cfg.seed)
env_info = collect_environment_info()
print('Loaded config:', cfg.run_name, '| config_hash:', cfg.config_hash())
print('Environment snapshot:', json.dumps(env_info, indent=2))

In [ ]:
#@title 6. Execute Pipeline (GPU Smoke Test OR Experiments A, B, C, D with Safe Resume)
import json
from bangla_smish.post_stage_b_runner import run_post_stage_b_pipeline

if RUN_MODE == 'gpu_smoke_test' and RUN_INTERRUPTION_RESUME_CHECK:
    print('--- Step 6a: Running 1 unit and simulating runtime interruption ---')
    partial = run_post_stage_b_pipeline(
        bundle_dir=BUNDLE_DIR,
        artifacts_dir=ARTIFACTS_DIR,
        cfg=cfg,
        resume=RESUME,
        verify_checksums=True,
        stop_after_units=1,
    )
    print('Completed after partial pass:', partial['completed_unit_count'])
    print('--- Step 6b: Resuming from persistent Drive checkpoint/state ---')

summary = run_post_stage_b_pipeline(
    bundle_dir=BUNDLE_DIR,
    artifacts_dir=ARTIFACTS_DIR,
    cfg=cfg,
    resume=RESUME,
    verify_checksums=True,
)
print('Completed units total:', summary['completed_unit_count'])

In [ ]:
#@title 7. Inspect Exported Results, Logs & Persistent Storage Artifacts
import pandas as pd
from pathlib import Path

exports_dir = ARTIFACTS_DIR / 'exports'
logs_path = ARTIFACTS_DIR / 'logs' / 'runner.log'
print('Exported files:', [p.name for p in sorted(exports_dir.glob('*'))])
print('Run state:', (ARTIFACTS_DIR / 'run_state.json').read_text(encoding='utf-8'))

csv_path = exports_dir / 'aggregated_results.csv'
if csv_path.exists() and csv_path.stat().st_size > 0:
    display_df = pd.read_csv(csv_path)
    print(display_df.to_string(index=False))

print('\nLast 15 log lines:')
if logs_path.exists():
    print('\n'.join(logs_path.read_text(encoding='utf-8').splitlines()[-15:]))